In [102]:
import asyncio
import logging
import os
from pathlib import Path
import yaml
import pandas as pd
from typing import Dict, Any, Tuple, List, Optional
from dotenv import load_dotenv
import argparse
from datetime import datetime

from src.factories.clothing_factory import ClothingFactory
from src.services.api_service import APIService
from src.services.attribute_generator import AttributeGenerator
from src.services.batch_processor import BatchProcessor
from src.utils.logging_utils import (
    setup_logging, 
    LoggingContext, 
    PerformanceMetrics,
    StructuredLogger
)
from src.services.product_standardizer import ProductStandardizer
from src.base.clothing_item import ClothingItem


In [103]:
from abc import ABC, abstractmethod
from enum import Enum
from typing import List, Dict, Any, Union
import re
import logging
from pydantic import (
    BaseModel, Field, field_validator, ValidationInfo
)
from src.base.enums import (
    Color, ColorDetailed, Pattern, Material, 
    EmbellishmentLevel, Embellishment, Occasion, 
    Style, Gender, AgeGroup
)
from src.utils.validation import validate_enum_field, validate_enum_list
from src.utils.config_loader import ConfigManager
import pandas as pd


In [104]:
class ConfigLoader:
    def __init__(self, config_dir: Path):
        self.config_dir = config_dir
        
    def load(self) -> Dict[str, Any]:
        """Load base configuration from YAML file"""
        config_path = self.config_dir / "base_config.yaml"
        with open(config_path) as f:
            return yaml.safe_load(f)


def load_config(config_path: str) -> Dict:
    with open(config_path) as f:
        return yaml.safe_load(f)

In [105]:
base_path = '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/product_attributes/config/base_config.yaml'
config = load_config(base_path)
from src.utils.config_loader import ConfigManager

## Load products

In [106]:
#!/usr/bin/env python3

import json
import os
from typing import Dict, Any, Optional


def float_hook(dct: dict) -> dict:
    for k, v in dct.items():
        if isinstance(v, str):
            try:
                if 'e' in v.lower():
                    dct[k] = float(v)
            except ValueError:
                pass
        elif isinstance(v, list):
            for i, item in enumerate(v):
                if isinstance(item, str) and 'e' in item.lower():
                    try:
                        v[i] = float(item)
                    except ValueError:
                        pass
    return dct


def count_unique_products(file_path: str) -> int:
    try:
        all_data = {}
        current_obj = ""
        brace_count = 0
        total_objects_found = 0
        failed_objects = 0
        
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                brace_count += line.count('{') - line.count('}')
                current_obj += line
                
                if brace_count == 0 and current_obj:
                    total_objects_found += 1
                    try:
                        data = json.loads(current_obj, object_hook=float_hook)
                        if data:
                            product_id = next(iter(data))
                            all_data[product_id] = data[product_id]
                    except json.JSONDecodeError:
                        failed_objects += 1
                        preview_len = 75
                        truncated = len(current_obj) > preview_len
                        preview = current_obj[:preview_len]
                        if truncated:
                            preview += "..."
                        obj_num = total_objects_found
                        print(f"\nFailed to parse object {obj_num}:")
                        print(preview)
                    current_obj = ""
        
        product_count = len(all_data)
        print("\nParsing Statistics:")
        print(f"Total JSON objects found: {total_objects_found}")
        print(f"Failed to parse: {failed_objects}")
        print(f"Unique products found: {product_count}")
        
        if product_count > 0:
            print("\nFirst few product IDs found:")
            for pid in list(all_data.keys())[:5]:
                print(f"- {pid}")
                
        return product_count
            
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return 0
    except Exception as e:
        print(f"Error: {str(e)}")
        return 0


def load_product_by_id(file_path: str, product_id: str) -> Optional[Dict[str, Any]]:
    try:
        current_obj = ""
        brace_count = 0
        
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                brace_count += line.count('{') - line.count('}')
                current_obj += line
                
                if brace_count == 0 and current_obj:
                    try:
                        data = json.loads(current_obj, object_hook=float_hook)
                        if data and product_id in data:
                            return data[product_id]
                    except json.JSONDecodeError:
                        pass
                    current_obj = ""
        
        print(f"Product ID {product_id} not found")
        return None
            
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"Error: {str(e)}")
        return None


json_path = '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/product_attributes/output/saree_attributes.json'
count = count_unique_products(json_path)
print(f"\nFinal count of unique products: {count}")


Parsing Statistics:
Total JSON objects found: 209
Failed to parse: 0
Unique products found: 209

First few product IDs found:
- 7372922781761
- 7372876546113
- 7372801835073
- 7372794265665
- 7372774015041

Final count of unique products: 209


In [108]:
product_temp = load_product_by_id(json_path, '7416278614081')
if product_temp:
    product_temp.pop('text_embedding', None)
    product_temp.pop('image_embedding', None)
product_temp

{'primary_color': 'Beige',
 'primary_color_hex': '#F5E1C9',
 'primary_color_detailed': 'Ecru',
 'secondary_colors': ['Red', 'Black'],
 'secondary_color_hexes': ['#D32F2F', '#000000'],
 'secondary_colors_detailed': ['Crimson', 'Jet Black'],
 'secondary_colors_detailed_hex': ['#DC143C', '#000000'],
 'color_pairings': ['#F5E1C9', '#D32F2F', '#000000', '#FFA07A', '#8B4513'],
 'pattern': ['Striped', 'Geometric'],
 'saree_type': 'Contemporary',
 'border_width': 'Narrow',
 'border_design': 'Embroidered',
 'border_design_details': ['Intricate geometric patterns',
  'Alternating red and black embroidery',
  'Delicate dotted lines',
  'Subtle contrast to main body'],
 'pallu_design': 'Contrast',
 'pre_draped': False,
 'material': 'Linen',
 'length': 6.3,
 'width': 1.1,
 'weight': 450,
 'care_instructions': 'Dry clean only. Store in a cool, dry place.',
 'occasions': ['Casual', 'Festive', 'Party'],
 'occasions_detailed': ['Art gallery openings',
  'Fusion cocktail parties',
  'Cultural festivals'

In [101]:
product_temp

{'primary_color': 'White',
 'primary_color_hex': '#FFFFFF',
 'primary_color_detailed': 'OffWhite',
 'secondary_colors': ['Red'],
 'secondary_color_hexes': ['#FF0000'],
 'secondary_colors_detailed': ['Crimson'],
 'secondary_colors_detailed_hex': ['#DC143C'],
 'color_pairings': ['#FF0000', '#FFA07A', '#FFB6C1', '#F08080', '#CD5C5C'],
 'pattern': ['Geometric', 'Abstract'],
 'saree_type': 'Traditional',
 'border_width': 'Narrow',
 'border_design': 'Printed',
 'border_design_details': ['Red geometric pattern',
  'Checkered design at bottom',
  'Abstract motifs along border',
  'Contrasting red on white base',
  'Seamless integration with body design'],
 'pallu_design': 'Printed',
 'pre_draped': False,
 'material': 'Cotton',
 'length': 5.5,
 'width': 1.1,
 'weight': 400,
 'care_instructions': 'Gentle machine wash, do not bleach, iron on medium heat',
 'occasions': ['Casual', 'Festive', 'Office Wear'],
 'occasions_detailed': ['Art gallery openings',
  'Cultural festivals like Durga Puja',
  '

In [75]:
product_type = 'saree'
ConfigManager.get_search_context_config(product_type)
# Load product configuration
product_config = ConfigManager.get_product_config(product_type)
attributes_config = product_config.get('attributes', {})

context_parts = []

# Process attributes based on in_search_context flag
for attr_name, attr_config in attributes_config.items():
    # Skip if attribute is not meant for search context
    if not isinstance(attr_config, dict) or not attr_config.get(
        'in_search_context', False
    ):
        continue
    
    # Skip if attribute doesn't exist in the object
    if not hasattr(dict, attr_name):
        continue
    
    value = getattr(dict, attr_name)
    
    # Skip empty values
    if value is None or (
        isinstance(value, (list, dict)) and not value
    ):
        continue
    
    # Format the attribute name for display
    display_name = attr_name.replace('_', ' ').title()
    
    # Process different types of attributes
    if isinstance(value, Enum):
        context_parts.append(f"{display_name}: {value.value}")
    
    elif isinstance(value, list):
        if all(isinstance(item, Enum) for item in value):
            items_str = ', '.join(item.value for item in value)
            if items_str:
                context_parts.append(
                    f"{display_name}: {items_str}"
                )
        elif all(isinstance(item, str) for item in value):
            items_str = ', '.join(item for item in value)
            if items_str:
                context_parts.append(
                    f"{display_name}: {items_str}"
                )
    
    elif isinstance(value, bool):
        if value:
            context_parts.append(f"{display_name}: Yes")
    
    elif isinstance(value, (str, int, float)):
        # Check for units in the attribute configuration
        if 'unit' in attr_config:
            context_parts.append(
                f"{display_name}: {value} {attr_config['unit']}"
            )
        # Fallback to hard-coded units for backward compatibility
        elif attr_name == 'length' and product_type == 'saree':
            context_parts.append(f"{display_name}: {value} meters")
        elif attr_name == 'length' and product_type == 'kurta':
            context_parts.append(f"{display_name}: {value} inches")
        elif attr_name == 'width':
            context_parts.append(f"{display_name}: {value} meters")
        elif attr_name == 'weight':
            context_parts.append(f"{display_name}: {value} grams")
        else:
            context_parts.append(f"{display_name}: {value}")
    
    elif isinstance(value, dict) and attr_name == 'coordinating_items':
        coord_parts = []
        for category, items in value.items():
            if items and isinstance(items, list):
                items_str = ', '.join(str(item) for item in items)
                if items_str:
                    coord_parts.append(f"{category}: {items_str}")
        
        if coord_parts:
            coord_str = '; '.join(coord_parts)
            context_parts.append(
                f"Coordinating Items: {coord_str}"
            )

# Filter out empty strings
context_parts = [p for p in context_parts if p and p.strip()]

# Join with periods and clean up extra spaces
search_context = '. '.join(context_parts).replace("  ", " ")

In [ ]:
def build_search_context(self) -> 'ClothingItem':
        """
        Build search context string from product attributes.
        This method uses configuration to determine which attributes 
        to include.
        
        Returns:
            ClothingItem: Self reference with updated search_context
        """
        # Get the product type from the class name
        product_type = self.__class__.__name__.lower()
        
        try:
            # Load product configuration
            product_config = ConfigManager.get_product_config(product_type)
            attributes_config = product_config.get('attributes', {})
            
            context_parts = []
            
            # Process attributes based on in_search_context flag
            for attr_name, attr_config in attributes_config.items():
                # Skip if attribute is not meant for search context
                if not isinstance(attr_config, dict) or not attr_config.get(
                    'in_search_context', False
                ):
                    continue
                
                # Skip if attribute doesn't exist in the object
                if not hasattr(self, attr_name):
                    continue
                
                value = getattr(self, attr_name)
                
                # Skip empty values
                if value is None or (
                    isinstance(value, (list, dict)) and not value
                ):
                    continue
                
                # Format the attribute name for display
                display_name = attr_name.replace('_', ' ').title()
                
                # Process different types of attributes
                if isinstance(value, Enum):
                    context_parts.append(f"{display_name}: {value.value}")
                
                elif isinstance(value, list):
                    if all(isinstance(item, Enum) for item in value):
                        items_str = ', '.join(item.value for item in value)
                        if items_str:
                            context_parts.append(
                                f"{display_name}: {items_str}"
                            )
                    elif all(isinstance(item, str) for item in value):
                        items_str = ', '.join(item for item in value)
                        if items_str:
                            context_parts.append(
                                f"{display_name}: {items_str}"
                            )
                
                elif isinstance(value, bool):
                    if value:
                        context_parts.append(f"{display_name}: Yes")
                
                elif isinstance(value, (str, int, float)):
                    # Check for units in the attribute configuration
                    if 'unit' in attr_config:
                        context_parts.append(
                            f"{display_name}: {value} {attr_config['unit']}"
                        )
                    # Fallback to hard-coded units for backward compatibility
                    elif attr_name == 'length' and product_type == 'saree':
                        context_parts.append(f"{display_name}: {value} meters")
                    elif attr_name == 'length' and product_type == 'kurta':
                        context_parts.append(f"{display_name}: {value} inches")
                    elif attr_name == 'width':
                        context_parts.append(f"{display_name}: {value} meters")
                    elif attr_name == 'weight':
                        context_parts.append(f"{display_name}: {value} grams")
                    else:
                        context_parts.append(f"{display_name}: {value}")
                
                elif isinstance(value, dict) and attr_name == 'coordinating_items':
                    coord_parts = []
                    for category, items in value.items():
                        if items and isinstance(items, list):
                            items_str = ', '.join(str(item) for item in items)
                            if items_str:
                                coord_parts.append(f"{category}: {items_str}")
                    
                    if coord_parts:
                        coord_str = '; '.join(coord_parts)
                        context_parts.append(
                            f"Coordinating Items: {coord_str}"
                        )
            
            # Filter out empty strings
            context_parts = [p for p in context_parts if p and p.strip()]
            
            # Join with periods and clean up extra spaces
            self.search_context = '. '.join(context_parts).replace("  ", " ")
            
        except Exception as e:
            logger.error(
                f"Error building search context for {product_type}: {str(e)}"
            )
            # Fallback to a minimal context
            self.search_context = (
                f"Title: {self.title}. Description: {self.description}"
            )
        
        return self


In [95]:
from src.models.saree import Saree
from src.base import (
    Color, ColorDetailed, Pattern, Material,
    EmbellishmentLevel, Embellishment, Occasion,
    Style, Gender, AgeGroup, SareeType,
    BorderWidth, BorderDesign, PalluDesign
)

# Create the test product instance
test_product = Saree(
    primary_color=Color.RED,
    primary_color_hex="#FF0000",
    primary_color_detailed=ColorDetailed.SCARLET,
    secondary_colors=[Color.ORANGE, Color.YELLOW, Color.GREEN],
    secondary_color_hexes=['#FFA500', '#FFFF00', '#008000'],
    secondary_colors_detailed=[
        ColorDetailed.TANGERINE, 
        ColorDetailed.LEMON,
        ColorDetailed.EMERALD
    ],
    secondary_colors_detailed_hex=['#FFA500', '#FFFF00', '#50C878'],
    color_pairings=['#FF0000', '#FFA500', '#FFFF00'],
    pattern=[Pattern.STRIPED],
    saree_type=SareeType.CONTEMPORARY,
    border_width=BorderWidth.NARROW,
    border_design=BorderDesign.PLAIN,
    border_design_details={
        'Thin border separating color blocks',
        'Clean lines between colors',
        'Minimalist edge design'
    },
    pallu_design=PalluDesign.CONTRAST,
    material=Material.OTHERS,
    length=5.5,
    width=1.14,
    weight=400,
    care_instructions="Hand wash separately in cold water and salt",
    occasions=[Occasion.CASUAL, Occasion.FESTIVAL],
    embellishment_level=EmbellishmentLevel.MINIMAL,
    embellishment=[Embellishment.NONE],
    style=[Style.CONTEMPORARY, Style.BOLD],
    brand_title="GAR-1967",
    title="Rainbow Rhapsody: A Vibrant Veil of Pride",
    description="Contemporary colorful saree for pride events",
    price=299.99,  # Required field added
    size=["One Size"],  # Required field added
    gender=[Gender.FEMALE],
    age_group=[AgeGroup.ADULT],
    coordinating_items={
        "clothing": ["White crop top", "Black fitted blouse"],
        "accessories": ["Rainbow gemstone necklace"],
        "footwear": ["Clear strap heeled sandals"]
    }
)

# Build and inspect the search context
test_product.build_search_context()
print("Generated Search Context:\n")
print(test_product.search_context)

AttributeError: SCARLET

In [81]:
Saree(**)

SyntaxError: invalid syntax (79317852.py, line 1)